<a href="https://colab.research.google.com/github/YonggunJung/Lotto/blob/main/1228Lotto%ED%95%9C%EC%A4%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from google.colab import files

In [3]:
# CSV 업로드
# uploaded = files.upload()
import io
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/로또/data/lotto_1line.csv', encoding='utf-8-sig')
df

,1,2,3,4,5,6,보너스
0,11,21,22,30,39,44,31
1,13,14,18,21,34,44,16
2,11,16,25,27,35,36,37
3,4,7,17,18,38,44,36
4,8,12,13,29,33,42,5
...,...,...,...,...,...,...,...
1222,16,18,20,32,33,39,26
1223,9,18,21,27,44,45,28
1224,8,9,19,25,41,42,33
1225,4,6,13,17,26,28,41


In [4]:
# 컬럼 이름 제외하고, 1행부터 마지막 행까지 리스트로 변환
lotto_list = df.values.flatten().tolist()

print(lotto_list[:20])   # 앞 5개만 확인
print(len(lotto_list))  # 전체 행 개수 확인

[11, 21, 22, 30, 39, 44, 31, 13, 14, 18, 21, 34, 44, 16, 11, 16, 25, 27, 35, 36]
8589


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [6]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# =========================================================
# 1. 컬럼 이름 제외하고, 전체 데이터를 1차원 리스트로 변환
# =========================================================

# df가 이미 만들어져 있다고 가정
# 예: df = pd.read_csv('lotto_1line.csv', encoding='utf-8-sig')

lotto_list = df.values.flatten().tolist()

# 혹시 문자열로 들어온 숫자가 있으면 정수로 변환
lotto_list = [int(x) for x in lotto_list if pd.notna(x)]

print("앞 20개 숫자:", lotto_list[:20])
print("전체 숫자 개수:", len(lotto_list))


# =========================================================
# 2. 학습 데이터 만들기
#    지금까지 나온 모든 숫자의 출현 횟수로 다음 숫자 1개 예측
# =========================================================

X = []
y = []

for i in range(1, len(lotto_list)):

    # i번째 숫자를 예측하기 위해 i 이전까지의 모든 숫자 사용
    previous_numbers = lotto_list[:i]

    # 실제 다음 숫자
    next_number = lotto_list[i]

    # 1~45 숫자의 출현 횟수 계산
    counts = np.bincount(previous_numbers, minlength=46)[1:46]

    X.append(counts)
    y.append(next_number)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


# =========================================================
# 3. 학습용 / 테스트용 데이터 분리
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    shuffle=False
)


# =========================================================
# 4. 모델 생성 및 학습
# =========================================================

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

print("모델 학습 완료")


# =========================================================
# 5. 테스트 정확도 확인
# =========================================================

pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)

print("정확도:", acc)


# =========================================================
# 6. 아무 숫자도 모르는 상태에서 이번 주 번호 6개 예측
# =========================================================

this_week = []

# 원본 lotto_list는 그대로 두고 예측용 리스트 복사
lotto_list1 = lotto_list.copy()

while len(this_week) < 6:

    # 현재까지의 모든 숫자 출현 횟수 계산
    all_counts = np.bincount(lotto_list1, minlength=46)[1:46]

    # 모델 입력 형태로 변경
    next_input = all_counts.reshape(1, -1)

    # 숫자별 예측 확률 계산
    proba = model.predict_proba(next_input)[0]
    classes = model.classes_

    # 확률표 만들기
    result = pd.DataFrame({
        'number': classes.astype(int),
        'probability': proba
    })

    # 이미 이번 주 번호에 들어간 숫자는 제외
    result = result[~result['number'].isin(this_week)]

    # 확률 높은 순서로 정렬
    result = result.sort_values(by='probability', ascending=False)

    # 가장 확률 높은 숫자 1개 선택
    next_number = int(result.iloc[0]['number'])

    # 이번 주 번호에 추가
    this_week.append(next_number)

    # 다음 번호 예측에 반영하기 위해 lotto_list1에도 추가
    lotto_list1.append(next_number)

    print(f"{len(this_week)}번째 예측 번호:", next_number)
    print("현재 this_week:", this_week)
    print(result.head(10))
    print("-" * 50)


print("최종 예측 번호 6개:", this_week)
print("정렬된 예측 번호:", sorted(this_week))

앞 20개 숫자: [11, 21, 22, 30, 39, 44, 31, 13, 14, 18, 21, 34, 44, 16, 11, 16, 25, 27, 35, 36]
전체 숫자 개수: 8589
X shape: (8588, 45)
y shape: (8588,)
모델 학습 완료
정확도: 0.02910360884749709
1번째 예측 번호: 6
현재 this_week: [6]
    number  probability
5        6     0.633333
4        5     0.230000
3        4     0.090000
0        1     0.033333
11      12     0.006667
43      44     0.003333
22      23     0.003333
1        2     0.000000
2        3     0.000000
8        9     0.000000
--------------------------------------------------
2번째 예측 번호: 5
현재 this_week: [6, 5]
    number  probability
4        5     0.230000
3        4     0.090000
0        1     0.033333
11      12     0.006667
43      44     0.003333
22      23     0.003333
2        3     0.000000
1        2     0.000000
8        9     0.000000
9       10     0.000000
--------------------------------------------------
3번째 예측 번호: 4
현재 this_week: [6, 5, 4]
    number  probability
3        4     0.090000
0        1     0.033333
11      12     0.00

In [7]:
[1, 4, 5, 6, 12, 44]

[7, 18, 22, 23, 29, 33]